# BGGTDM v2 — Prior Season Carry-Forward State

Builds two lookup tables used by `early_season.py` for weeks 1-3 predictions:

- `data/prior_season_final_state.csv` — each player's end-of-season feature values, joined into the following season as week 1-3 carry-forward features
- `data/rookie_buckets.csv` — median week 1-3 stats by draft_round + pos, used when a player has no prior-season row

Run once after retraining. Re-run at the start of each new season to pick up the latest season's final state.

In [1]:
import os
os.chdir('/Users/aaronangeles/Documents/Projects/Active/biggamegabefallacy/ml')

import numpy as np
import pandas as pd
import joblib

from feature_prep import add_target_share, shrink_td_rate, shrink_rz_td_rate

pd.options.display.max_columns = 40
pd.options.display.float_format = '{:.4f}'.format

df = pd.read_csv('data/game_logs_enriched.csv')
print(f'Loaded {len(df)} rows | seasons: {sorted(df["season"].unique())}')
print(f'Columns: {list(df.columns)}')

Loaded 9972 rows | seasons: [np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
Columns: ['player_id', 'name', 'pos', 'team', 'game_id', 'season', 'week', 'receptions', 'targets', 'rec_yards', 'rec_tds', 'long_rec', 'rec_avg', 'scored_td', 'target_bucket', 'lag_yards', 'lag_targets', 'lag_receptions', 'cum_yards', 'cum_targets', 'cum_receptions', 'cum_tds', 'games_played', 'yards_pg', 'targets_pg', 'receptions_pg', 'roll3_yards', 'roll3_targets', 'roll3_receptions', 'td_rate_per_target', 'lag_ypr', 'target_trend', 'weeks_since_td', 'is_te', 'is_week1', 'exp', 'catch_rate_raw', 'lag_catch_rate', 'roll3_catch_rate', 'season_catch_rate', 'lag_ypt', 'season_ypt', 'roll3_long_rec', 'cr_q', 'roll3_target_std', 'target_cv', 'season_target_std', 'td_streak', 'td_last1', 'td_last2', 'td_last3', 'tds_last3', 'height_in', 'weight', 'exp_tier', 'height_bucket', 'is_home', 'opp_wr_te_td_rate_allowed', 'opp_wr_te_targets_pg_allowed', 'lag_snap_pct', 'roll3_snap_pct', 'season_snap_pct', 

In [2]:
# Add target_share (computed from cum_targets, not stored in CSV)
df = add_target_share(df)
print(f'target_share null rate: {df["target_share"].isna().mean():.1%}')

target_share null rate: 0.0%


In [3]:
# Fit EB shrinkage on training data only (seasons <= 2024, games_played >= 1)
# then apply to all rows — same approach as feature_prep.load_and_prepare()
train_mask = (df['season'] <= 2024) & (df['games_played'] >= 1)
train_rows = df[train_mask].copy()

# Anytime TD rate EB
_, alpha_eb, beta_eb = shrink_td_rate(train_rows)
df, _, _ = shrink_td_rate(df, alpha=alpha_eb, beta=beta_eb)
print(f'EB params: alpha={alpha_eb:.4f}, beta={beta_eb:.4f}')

# Red zone TD rate EB (only if RZ columns are present)
rz_alpha, rz_beta = None, None
if 'cum_rz_tds' in df.columns:
    _, rz_alpha, rz_beta = shrink_rz_td_rate(train_rows)
    df, _, _ = shrink_rz_td_rate(df, alpha=rz_alpha, beta=rz_beta)
    print(f'RZ EB params: alpha={rz_alpha:.4f}, beta={rz_beta:.4f}')
else:
    print('NOTE: RZ columns not found — skipping rz_td_rate_eb')

print(f'td_rate_eb sample: {df["td_rate_eb"].describe()}')

Fitting Beta-Binomial parameters...


  alpha=17.4431, beta=339.8969
EB params: alpha=17.4431, beta=339.8969
Fitting Beta-Binomial parameters for RZ TD rate...
  RZ EB: alpha=69.6489, beta=211.5923
RZ EB params: alpha=69.6489, beta=211.5923
td_rate_eb sample: count   9971.0000
mean       0.0489
std        0.0033
min        0.0365
25%        0.0470
50%        0.0484
75%        0.0505
max        0.0689
Name: td_rate_eb, dtype: float64


In [4]:
# Fetch draft_round from nfl_data_py (needed for rookie bucket assignment)
import nfl_data_py as nfl

ids = nfl.import_ids()
print(f'ID table: {len(ids)} rows, columns: {list(ids.columns)}')

# Build name_norm -> draft_round lookup (take most recent season row per player)
ids_clean = ids[ids['draft_round'].notna()][['name', 'draft_round', 'position']].copy()
ids_clean['name_norm'] = ids_clean['name'].str.lower().str.strip()
ids_clean['draft_round'] = ids_clean['draft_round'].astype(int)

# Deduplicate: keep first occurrence per normalized name
ids_clean = ids_clean.drop_duplicates(subset='name_norm', keep='first')
draft_lookup = ids_clean.set_index('name_norm')['draft_round'].to_dict()

df['name_norm'] = df['name'].str.lower().str.strip()
df['draft_round'] = df['name_norm'].map(draft_lookup).fillna(0).astype(int)
df.drop(columns=['name_norm'], inplace=True)

# draft_round == 0 means undrafted or not found — treat as lowest bucket
print(f'Players with draft_round > 0: {(df["draft_round"] > 0).mean():.1%}')
print(df['draft_round'].value_counts().sort_index())

ID table: 12187 rows, columns: ['twitter_username', 'birthdate', 'college', 'rotowire_id', 'draft_pick', 'weight', 'pff_id', 'sportradar_id', 'draft_round', 'rotoworld_id', 'cfbref_id', 'swish_id', 'stats_id', 'ktc_id', 'espn_id', 'team', 'fleaflicker_id', 'nfl_id', 'pfr_id', 'stats_global_id', 'fantasy_data_id', 'age', 'db_season', 'position', 'gsis_id', 'height', 'mfl_id', 'merge_name', 'draft_ovr', 'fantasypros_id', 'cbs_id', 'yahoo_id', 'sleeper_id', 'name', 'draft_year']
Players with draft_round > 0: 80.5%
draft_round
0    1942
1    1752
2    1910
3    1355
4    1284
5     854
6     622
7     253
Name: count, dtype: int64


In [5]:
# Define the carry-forward feature columns — must match CARRY_FEATURES in early_season.py
CARRY_COLS = [
    # Volume
    'targets_pg', 'yards_pg', 'receptions_pg',
    'roll3_targets', 'roll3_yards', 'roll3_receptions',
    'lag_targets', 'lag_yards',
    # Share
    'target_share',
    # Downfield / volatility
    'roll3_long_rec', 'roll3_target_std',
    # Streak / momentum
    'tds_last3', 'td_streak',
    # EB rates
    'td_rate_eb', 'td_rate_eb_std',
    # Snap %
    'lag_snap_pct', 'roll3_snap_pct',
    # Red zone
    'roll3_rz_targets', 'rz_target_share', 'rz_td_rate_eb',
]

# Only keep columns that actually exist (handles case where 05_new_features hasn't been run)
CARRY_COLS = [c for c in CARRY_COLS if c in df.columns]
print(f'Carry columns ({len(CARRY_COLS)}): {CARRY_COLS}')

Carry columns (20): ['targets_pg', 'yards_pg', 'receptions_pg', 'roll3_targets', 'roll3_yards', 'roll3_receptions', 'lag_targets', 'lag_yards', 'target_share', 'roll3_long_rec', 'roll3_target_std', 'tds_last3', 'td_streak', 'td_rate_eb', 'td_rate_eb_std', 'lag_snap_pct', 'roll3_snap_pct', 'roll3_rz_targets', 'rz_target_share', 'rz_td_rate_eb']


In [6]:
# ---- Build prior_season_final_state ----
# Take each player's last game row of each season as carry-forward state
# Include all seasons so we always have the most recent prior season available

prior_final = (
    df.sort_values(['player_id', 'season', 'week'])
    .groupby(['player_id', 'season'])
    .last()
    .reset_index()
)

keep_cols = ['player_id', 'season', 'name', 'pos', 'team', 'draft_round', 'is_te'] + CARRY_COLS
keep_cols = [c for c in keep_cols if c in prior_final.columns]
prior_final = prior_final[keep_cols].copy()

# join_season: the season this row acts as carry-forward for
# e.g. a player's 2024 final state joins into 2025 week 1
prior_final['join_season'] = prior_final['season'] + 1

print(f'prior_season_final_state: {len(prior_final)} rows')
print(f'Seasons covered: {sorted(prior_final["season"].unique())}')
print(f'join_seasons: {sorted(prior_final["join_season"].unique())}')
prior_final.head(5)

prior_season_final_state: 966 rows
Seasons covered: [np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
join_seasons: [np.int64(2023), np.int64(2024), np.int64(2025), np.int64(2026)]


,player_id,season,name,pos,team,draft_round,is_te,targets_pg,yards_pg,receptions_pg,roll3_targets,roll3_yards,roll3_receptions,lag_targets,lag_yards,target_share,roll3_long_rec,roll3_target_std,tds_last3,td_streak,td_rate_eb,td_rate_eb_std,lag_snap_pct,roll3_snap_pct,roll3_rz_targets,rz_target_share,rz_td_rate_eb,join_season
0,15795,2022,DeAndre Hopkins,WR,ARI,1,0,10.7500,89.1250,7.8750,9.3333,75.3333,6.0000,11.0000,60.0000,0.3660,24.6667,2.8868,1.0000,0,0.0461,0.0099,0.9200,0.8367,0.0000,0.0000,0.2476,2023
1,15795,2023,DeAndre Hopkins,WR,TEN,1,0,7.9375,63.1875,4.2500,6.6667,37.6667,3.6667,7.0000,72.0000,0.5474,16.0000,2.5166,0.0000,0,0.0484,0.0097,0.8500,0.8167,6.0000,0.3542,0.2436,2024
2,15795,2024,DeAndre Hopkins,WR,KC,1,0,5.0667,40.2000,3.6000,6.3333,35.0000,4.3333,4.0000,37.0000,0.1944,12.3333,2.5166,1.0000,0,0.0518,0.0106,0.4900,0.4967,5.0000,0.0860,0.2546,2025
3,15795,2025,DeAndre Hopkins,WR,BAL,1,0,2.3125,20.6250,1.3750,2.3333,24.3333,1.6667,1.0000,0.0000,0.1440,16.0000,2.3094,0.0000,0,0.0493,0.0109,0.2500,0.3100,0.0000,0.0000,0.2476,2026
4,15818,2022,Keenan Allen,WR,LAC,3,0,8.6667,72.2222,6.4444,9.6667,83.3333,8.0000,6.0000,60.0000,0.3133,28.0000,4.0415,0.0000,0,0.0447,0.0099,0.8400,0.8567,7.0000,0.1383,0.2401,2023


In [7]:
# Sanity check: spot-check a known player's carry-forward state
# Should reflect their final 2024 season stats
test_name = 'Ja\'Marr Chase'
check = prior_final[
    (prior_final['name'] == test_name) &
    (prior_final['season'] == 2024)
]
if len(check) > 0:
    print(f"{test_name} 2024 final state (joins into 2025):")
    print(check[['name', 'season', 'join_season', 'targets_pg', 'td_rate_eb',
                  'rz_target_share', 'roll3_snap_pct']].to_string(index=False))
else:
    print(f'{test_name} not found')

Ja'Marr Chase 2024 final state (joins into 2025):
         name  season  join_season  targets_pg  td_rate_eb  rz_target_share  roll3_snap_pct
Ja'Marr Chase    2024         2025     10.0625      0.0645           0.2920          0.9467


In [8]:
# ---- Build rookie_buckets ----
# Median week 1-3 stats by draft_round + pos from historical training data
# Used when a player has no prior-season row at all

# Use training data (seasons <= 2024) early weeks to compute buckets
early_weeks = df[
    (df['season'] <= 2024) &
    (df['week'] <= 3) &
    (df['games_played'] >= 1)  # need at least 1 game to have any signal
].copy()

print(f'Early-week training rows: {len(early_weeks)}')
print(f'Unique draft_round values: {sorted(early_weeks["draft_round"].unique())}')

stat_cols = [c for c in CARRY_COLS if c != 'rz_td_rate_eb']  # rz_td_rate_eb needs cum cols

rookie_buckets = (
    early_weeks
    .groupby(['draft_round', 'pos'])[stat_cols]
    .median()
    .reset_index()
)

# If rz_td_rate_eb is in CARRY_COLS, compute it from medians of cum_rz_tds/cum_rz_targets
if 'rz_td_rate_eb' in CARRY_COLS and 'cum_rz_tds' in early_weeks.columns:
    rz_cum = (
        early_weeks
        .groupby(['draft_round', 'pos'])[['cum_rz_tds', 'cum_rz_targets']]
        .median()
        .reset_index()
    )
    if rz_alpha is not None:
        rz_cum['rz_td_rate_eb'] = (
            (rz_cum['cum_rz_tds'] + rz_alpha) /
            (rz_cum['cum_rz_targets'] + rz_alpha + rz_beta)
        )
    else:
        rz_cum['rz_td_rate_eb'] = 0.0
    rookie_buckets = rookie_buckets.merge(
        rz_cum[['draft_round', 'pos', 'rz_td_rate_eb']],
        on=['draft_round', 'pos'],
        how='left'
    )

print(f'rookie_buckets: {len(rookie_buckets)} rows')
print(rookie_buckets[['draft_round', 'pos', 'targets_pg', 'td_rate_eb']].to_string(index=False))

Early-week training rows: 702
Unique draft_round values: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7)]
rookie_buckets: 15 rows
 draft_round pos  targets_pg  td_rate_eb
           0  TE      3.0000      0.0484
           0  WR      4.0000      0.0485
           1  TE      4.0000      0.0480
           1  WR      6.0000      0.0484
           2  TE      4.0000      0.0483
           2  WR      5.5000      0.0483
           3  TE      3.0000      0.0484
           3  WR      4.0000      0.0484
           4  TE      3.0000      0.0483
           4  WR      4.0000      0.0484
           5  TE      2.0000      0.0485
           5  WR      3.0000      0.0484
           6  TE      4.5000      0.0484
           6  WR      2.5000      0.0485
           7  WR      3.0000      0.0483


In [9]:
# Sanity check: 1st-round WR bucket should be higher than 7th-round WR
wr_buckets = rookie_buckets[rookie_buckets['pos'] == 'WR'].sort_values('draft_round')
print('WR buckets by draft round:')
display_cols = ['draft_round', 'targets_pg', 'target_share', 'td_rate_eb']
display_cols = [c for c in display_cols if c in wr_buckets.columns]
print(wr_buckets[display_cols].to_string(index=False))

WR buckets by draft round:
 draft_round  targets_pg  target_share  td_rate_eb
           0      4.0000        0.1500      0.0485
           1      6.0000        0.2955      0.0484
           2      5.5000        0.2105      0.0483
           3      4.0000        0.1667      0.0484
           4      4.0000        0.1667      0.0484
           5      3.0000        0.1455      0.0484
           6      2.5000        0.1020      0.0485
           7      3.0000        0.1360      0.0483


In [10]:
# ---- Save both tables ----
prior_final.to_csv('data/prior_season_final_state.csv', index=False)
print(f'Saved data/prior_season_final_state.csv — {len(prior_final)} rows')

rookie_buckets.to_csv('data/rookie_buckets.csv', index=False)
print(f'Saved data/rookie_buckets.csv — {len(rookie_buckets)} rows')

Saved data/prior_season_final_state.csv — 966 rows
Saved data/rookie_buckets.csv — 15 rows


## Quick Usage Test

Demonstrates how `early_season.py` will use these tables for a simulated week 1.

In [11]:
from early_season import load_early_season_priors, resolve_early_features

prior, buckets = load_early_season_priors()

# Simulate a small 2025 week 1 player list:
# - A known vet (same team) → carry-forward
# - A team-changer (if applicable)
# - A rookie (no prior row)
sim_players = pd.DataFrame([
    # Known vet — use a player who was in 2024 data
    {'player_id': df[df['name'] == "Ja'Marr Chase"]['player_id'].iloc[0],
     'name': "Ja'Marr Chase", 'pos': 'WR', 'team': 'CIN', 'is_te': 0, 'draft_round': 1},
    # Fictional rookie
    {'player_id': 'ROOKIE_TEST_001',
     'name': 'Test Rookie', 'pos': 'WR', 'team': 'KC', 'is_te': 0, 'draft_round': 1},
])

result = resolve_early_features(
    players_df=sim_players,
    season=2025,
    week=1,
    prior_df=prior,
    bucket_df=buckets,
)

print('\nResolution result:')
show_cols = ['name', 'resolution', 'targets_pg', 'td_rate_eb', 'rz_target_share']
show_cols = [c for c in show_cols if c in result.columns]
print(result[show_cols].to_string(index=False))

Week 1 resolution: 1 carry-forward | 0 team-changers | 1 rookies/no-prior

Resolution result:
         name    resolution  targets_pg  td_rate_eb  rz_target_share
Ja'Marr Chase carry_forward     10.0625      0.0645           0.2920
  Test Rookie        rookie      6.0000      0.0484           0.0000
